<a href="https://colab.research.google.com/github/kcymae/Computational-TCell-Epitope-Analysis/blob/main/4_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ====================================================================
# NOTEBOOK 4: MACHINE LEARNING MODEL FOR EPITOPE PREDICTION
# Computational Identification of T-cell Epitopes using ML
# ====================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve,
                             auc, roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef, roc_auc_score)
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("=" * 70)
print("MACHINE LEARNING MODEL FOR EPITOPE PREDICTION - NOTEBOOK 4")
print("=" * 70)

# ====================================================================
# STEP 1: LOAD AND PREPARE DATA
# ====================================================================

print("\n" + "=" * 70)
print("STEP 1: LOADING AND PREPARING DATA")
print("=" * 70)

# Load data from GitHub
github_url = "https://raw.githubusercontent.com/kcymae/Computational-TCell-Epitope-Analysis/main/"

try:
    # Load combined epitope data
    combined_df = pd.read_csv(github_url + "epitope_prediction_combined.csv")
    print("✅ Loaded epitope data from GitHub")

except Exception as e:
    print(f"Loading from local file...")
    combined_df = pd.read_csv("epitope_prediction_combined.csv")

print(f"\n📊 Dataset Overview:")
print(f"   • Total epitopes: {len(combined_df)}")
print(f"   • Columns: {combined_df.columns.tolist()}")

# Load docking results (if available)
try:
    docking_df = pd.read_csv("docking_results.csv")
    print("✅ Loaded docking results")
    has_docking = True
except:
    print("⚠️ Docking results not found, using IEDB predictions only")
    has_docking = False
    docking_df = None

# ====================================================================
# STEP 2: FEATURE ENGINEERING
# ====================================================================

print("\n" + "=" * 70)
print("STEP 2: FEATURE ENGINEERING - CALCULATING PHYSICOCHEMICAL PROPERTIES")
print("=" * 70)

# Define amino acid properties
aa_properties = {
    # Hydrophobicity (Kyte-Doolittle scale)
    'hydrophobicity': {
        'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
        'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
        'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
        'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
    },

    # Charge at pH 7
    'charge': {
        'K': 1.0, 'R': 1.0, 'H': 0.1,  # Positive
        'D': -1.0, 'E': -1.0,  # Negative
        'A': 0, 'N': 0, 'C': 0, 'Q': 0, 'G': 0, 'I': 0,
        'L': 0, 'M': 0, 'F': 0, 'P': 0, 'S': 0, 'T': 0, 'W': 0, 'Y': 0, 'V': 0
    },

    # Molecular weight
    'mass': {
        'A': 89, 'R': 174, 'N': 132, 'D': 133, 'C': 121,
        'Q': 146, 'E': 147, 'G': 75, 'H': 155, 'I': 131,
        'L': 131, 'K': 146, 'M': 149, 'F': 165, 'P': 115,
        'S': 105, 'T': 119, 'W': 204, 'Y': 181, 'V': 117
    },

    # Aromaticity (presence of aromatic ring)
    'aromaticity': {
        'F': 1, 'Y': 1, 'W': 1,
        'A': 0, 'R': 0, 'N': 0, 'D': 0, 'C': 0, 'Q': 0, 'E': 0, 'G': 0, 'H': 0,
        'I': 0, 'L': 0, 'K': 0, 'M': 0, 'P': 0, 'S': 0, 'T': 0, 'V': 0
    }
}

# Function to calculate sequence features
def calculate_sequence_features(peptide_sequence):
    """Calculate physicochemical properties for a peptide"""

    features = {}

    # Basic properties
    features['length'] = len(peptide_sequence)
    features['hydrophobic_aa'] = sum(1 for aa in peptide_sequence if aa in 'AILMFVPW')
    features['aromatic_aa'] = sum(1 for aa in peptide_sequence if aa in 'FYW')
    features['polar_aa'] = sum(1 for aa in peptide_sequence if aa in 'STNQ')
    features['charged_aa'] = sum(1 for aa in peptide_sequence if aa in 'DEKR')
    features['positive_charged'] = sum(1 for aa in peptide_sequence if aa in 'KR')
    features['negative_charged'] = sum(1 for aa in peptide_sequence if aa in 'DE')

    # Calculate average hydrophobicity
    hydro_values = [aa_properties['hydrophobicity'].get(aa, 0) for aa in peptide_sequence]
    features['avg_hydrophobicity'] = np.mean(hydro_values) if hydro_values else 0
    features['max_hydrophobicity'] = np.max(hydro_values) if hydro_values else 0
    features['min_hydrophobicity'] = np.min(hydro_values) if hydro_values else 0

    # Calculate net charge
    charge_values = [aa_properties['charge'].get(aa, 0) for aa in peptide_sequence]
    features['net_charge'] = sum(charge_values)
    features['avg_charge'] = np.mean(charge_values) if charge_values else 0

    # Calculate molecular weight
    mass_values = [aa_properties['mass'].get(aa, 100) for aa in peptide_sequence]
    features['total_mass'] = sum(mass_values)
    features['avg_mass'] = np.mean(mass_values) if mass_values else 0

    # Aromaticity ratio
    features['aromaticity_ratio'] = features['aromatic_aa'] / features['length']
    features['hydrophobicity_ratio'] = features['hydrophobic_aa'] / features['length']

    # MHC anchor preferences for HLA-A*02:01
    # Position 2: P (Proline) or Hydrophobic (L, I, M, F, V, W)
    # Position 9 (C-terminus): V, I, L, M, F, W, Y, A
    mhc_p2_preference = 1 if len(peptide_sequence) > 1 and peptide_sequence[1] in 'LIMFVW' else 0
    mhc_p9_preference = 1 if len(peptide_sequence) > 8 and peptide_sequence[-1] in 'VILMFWYA' else 0
    features['mhc_p2_anchor'] = mhc_p2_preference
    features['mhc_p9_anchor'] = mhc_p9_preference

    return features

print(f"\n🧬 Calculating physicochemical features for {len(combined_df)} peptides...")

# Calculate features for all peptides
peptide_features_list = []
for idx, row in combined_df.iterrows():
    peptide = row['peptide']
    features = calculate_sequence_features(peptide)
    features['peptide'] = peptide
    features['allele'] = row['allele']
    features['iedb_score'] = row['score']
    features['percentile_rank'] = row['percentile_rank']
    peptide_features_list.append(features)

features_df = pd.DataFrame(peptide_features_list)

print(f"✅ Features calculated for all peptides")
print(f"\n📋 Feature Summary:")
print(f"   • Total features: {len(features_df.columns) - 4}")  # Exclude peptide, allele, scores
feature_cols = [col for col in features_df.columns if col not in ['peptide', 'allele', 'iedb_score', 'percentile_rank']]
print(f"   • Feature list: {feature_cols}")

# If docking data available, merge it
if has_docking and docking_df is not None:
    print(f"\n🔬 Merging docking results...")
    docking_merge = docking_df[['Peptide', 'Predicted_Binding_Energy_kcal_mol', 'Binding_Strength']].copy()
    docking_merge.columns = ['peptide', 'binding_energy', 'binding_strength']

    features_df = features_df.merge(docking_merge, on='peptide', how='left')

    # Fill missing docking values
    if features_df['binding_energy'].isnull().any():
        # For peptides without docking data, predict based on IEDB score
        features_df['binding_energy'].fillna(
            -7.5 - 5 * np.log10(features_df['iedb_score'].fillna(1.0)),
            inplace=True
        )

    print(f"✅ Docking data merged")
else:
    print(f"\n⚠️ Docking data not available, using IEDB score for binding prediction")
    features_df['binding_energy'] = -7.5 - 5 * np.log10(features_df['iedb_score'])

# ====================================================================
# STEP 3: CREATE TARGET VARIABLE (BINDER CLASSIFICATION)
# ====================================================================

print("\n" + "=" * 70)
print("STEP 3: CREATING TARGET VARIABLE - BINDER CLASSIFICATION")
print("=" * 70)

# Classification: Strong binder (1) vs Weak binder (0)
# Threshold: IEDB score < 0.5 nM = Strong binder
features_df['is_strong_binder'] = (features_df['iedb_score'] < 0.5).astype(int)

# Alternative classification based on binding energy
features_df['is_strong_binder_docking'] = (features_df['binding_energy'] < -8.0).astype(int)

# Use IEDB-based classification as primary target
target_col = 'is_strong_binder'

# Show class distribution
class_dist = features_df[target_col].value_counts()
print(f"\n📊 Target Variable Distribution:")
print(f"   • Strong binders (score < 0.5 nM): {class_dist[1]} ({class_dist[1]/len(features_df)*100:.1f}%)")
print(f"   • Weak binders (score >= 0.5 nM): {class_dist[0]} ({class_dist[0]/len(features_df)*100:.1f}%)")

# Display feature statistics
print(f"\n📈 Feature Statistics:")
feature_cols = [col for col in features_df.columns if col not in ['peptide', 'allele', 'iedb_score',
                                                                     'percentile_rank', 'binding_energy',
                                                                     'binding_strength', 'is_strong_binder',
                                                                     'is_strong_binder_docking']]
print(features_df[feature_cols].describe().round(3))

# ====================================================================
# STEP 4: PREPARE DATA FOR MACHINE LEARNING
# ====================================================================

print("\n" + "=" * 70)
print("STEP 4: PREPARING DATA FOR MACHINE LEARNING")
print("=" * 70)

# Select features for training (exclude identifiers and target)
X = features_df[feature_cols].copy()
y = features_df[target_col].copy()

print(f"\n📊 Data Preparation:")
print(f"   • Features (X): {X.shape}")
print(f"   • Target (y): {y.shape}")
print(f"   • Feature matrix shape: {X.shape[1]} features × {X.shape[0]} samples")

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\n📋 Train-Test Split:")
print(f"   • Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   • Testing set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"   • Training set binder distribution:")
print(f"      - Strong binders: {sum(y_train)} ({sum(y_train)/len(y_train)*100:.1f}%)")
print(f"      - Weak binders: {len(y_train)-sum(y_train)} ({(len(y_train)-sum(y_train))/len(y_train)*100:.1f}%)")

# Standardize features (important for many ML algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Features standardized (mean=0, std=1)")

# ====================================================================
# STEP 5: TRAIN MULTIPLE ML MODELS
# ====================================================================

print("\n" + "=" * 70)
print("STEP 5: TRAINING MULTIPLE MACHINE LEARNING MODELS")
print("=" * 70)

# Dictionary to store trained models
models = {}
results = []

# Model 1: Random Forest
print(f"\n🌳 Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_pred_proba = rf_model.predict_proba(X_test_scaled)[:, 1]
models['Random Forest'] = rf_model

print(f"   ✅ Trained")

# Model 2: Gradient Boosting
print(f"\n📊 Training Gradient Boosting Classifier...")
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5, learning_rate=0.1)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
gb_pred_proba = gb_model.predict_proba(X_test_scaled)[:, 1]
models['Gradient Boosting'] = gb_model

print(f"   ✅ Trained")

# Model 3: Support Vector Machine
print(f"\n🎯 Training Support Vector Machine...")
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)
svm_pred_proba = svm_model.predict_proba(X_test_scaled)[:, 1]
models['SVM'] = svm_model

print(f"   ✅ Trained")

# Model 4: Neural Network
print(f"\n🧠 Training Neural Network (MLP)...")
nn_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
nn_model.fit(X_train_scaled, y_train)
nn_pred = nn_model.predict(X_test_scaled)
nn_pred_proba = nn_model.predict_proba(X_test_scaled)[:, 1]
models['Neural Network'] = nn_model

print(f"   ✅ Trained")

print(f"\n✅ All models trained successfully!")

# ====================================================================
# STEP 6: EVALUATE MODEL PERFORMANCE
# ====================================================================

print("\n" + "=" * 70)
print("STEP 6: EVALUATING MODEL PERFORMANCE")
print("=" * 70)

# Evaluation function
def evaluate_model(y_test, y_pred, y_pred_proba, model_name):
    """Calculate performance metrics"""

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    return {
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'MCC': mcc,
        'ROC-AUC': roc_auc
    }

# Evaluate all models
print(f"\n📊 Model Performance Metrics:\n")

results_data = []
results_data.append(evaluate_model(y_test, rf_pred, rf_pred_proba, 'Random Forest'))
results_data.append(evaluate_model(y_test, gb_pred, gb_pred_proba, 'Gradient Boosting'))
results_data.append(evaluate_model(y_test, svm_pred, svm_pred_proba, 'SVM'))
results_data.append(evaluate_model(y_test, nn_pred, nn_pred_proba, 'Neural Network'))

results_df = pd.DataFrame(results_data)

print(results_df.to_string(index=False))

# Identify best model
best_model_idx = results_df['ROC-AUC'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
best_roc_auc = results_df.loc[best_model_idx, 'ROC-AUC']

print(f"\n🏆 BEST MODEL: {best_model_name} (ROC-AUC: {best_roc_auc:.4f})")

# Save results
results_df.to_csv("ml_model_performance.csv", index=False)
print(f"✅ Results saved to: ml_model_performance.csv")

# ====================================================================
# STEP 7: CROSS-VALIDATION ANALYSIS
# ====================================================================

print("\n" + "=" * 70)
print("STEP 7: CROSS-VALIDATION ANALYSIS")
print("=" * 70)

cv_results = {}

print(f"\n🔄 Performing 5-fold cross-validation...\n")

for model_name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
    cv_results[model_name] = cv_scores

    print(f"{model_name}:")
    print(f"   • Fold scores: {[f'{s:.4f}' for s in cv_scores]}")
    print(f"   • Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})\n")

# ====================================================================
# STEP 8: FEATURE IMPORTANCE ANALYSIS
# ====================================================================

print("\n" + "=" * 70)
print("STEP 8: FEATURE IMPORTANCE ANALYSIS")
print("=" * 70)

# Get feature importance from Random Forest and Gradient Boosting
rf_importance = rf_model.feature_importances_
gb_importance = gb_model.feature_importances_

# Create feature importance dataframe
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'RF_Importance': rf_importance,
    'GB_Importance': gb_importance
})

importance_df['Avg_Importance'] = (importance_df['RF_Importance'] + importance_df['GB_Importance']) / 2
importance_df = importance_df.sort_values('Avg_Importance', ascending=False)

print(f"\n📊 Top 15 Most Important Features:\n")
print(importance_df.head(15).to_string(index=False))

# Save feature importance
importance_df.to_csv("feature_importance.csv", index=False)
print(f"\n✅ Feature importance saved to: feature_importance.csv")

# ====================================================================
# STEP 9: VISUALIZATION - MODEL PERFORMANCE
# ====================================================================

print("\n" + "=" * 70)
print("STEP 9: VISUALIZING MODEL PERFORMANCE")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Machine Learning Model Performance Comparison', fontsize=16, fontweight='bold')

# 1. ROC Curves
ax1 = axes[0, 0]

model_predictions = {
    'Random Forest': rf_pred_proba,
    'Gradient Boosting': gb_pred_proba,
    'SVM': svm_pred_proba,
    'Neural Network': nn_pred_proba
}

colors = ['blue', 'green', 'red', 'purple']

for (model_name, y_proba), color in zip(model_predictions.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=color, lw=2, label=f'{model_name} (AUC={roc_auc:.3f})')

ax1.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
ax1.set_xlabel('False Positive Rate', fontweight='bold')
ax1.set_ylabel('True Positive Rate', fontweight='bold')
ax1.set_title('ROC Curves - Model Comparison', fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(alpha=0.3)

# 2. Model Performance Metrics
ax2 = axes[0, 1]

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.2

for i, model_name in enumerate(results_df['Model']):
    values = results_df[results_df['Model'] == model_name][metrics_to_plot].values[0]
    ax2.bar(x + i*width, values, width, label=model_name, alpha=0.8)

ax2.set_xlabel('Metrics', fontweight='bold')
ax2.set_ylabel('Score', fontweight='bold')
ax2.set_title('Model Performance Comparison', fontweight='bold')
ax2.set_xticks(x + width * 1.5)
ax2.set_xticklabels(metrics_to_plot, rotation=45, ha='right')
ax2.legend()
ax2.set_ylim([0, 1.05])
ax2.grid(alpha=0.3, axis='y')

# 3. Feature Importance (Top 15)
ax3 = axes[1, 0]

top_features = importance_df.head(15)
ax3.barh(range(len(top_features)), top_features['Avg_Importance'], color='steelblue', alpha=0.7, edgecolor='black')
ax3.set_yticks(range(len(top_features)))
ax3.set_yticklabels(top_features['Feature'], fontsize=10)
ax3.set_xlabel('Average Importance', fontweight='bold')
ax3.set_title('Top 15 Most Important Features', fontweight='bold')
ax3.invert_yaxis()
ax3.grid(alpha=0.3, axis='x')

# 4. Confusion Matrix (Best Model - Random Forest)
ax4 = axes[1, 1]

cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4, cbar_kws={'label': 'Count'},
            xticklabels=['Weak Binder', 'Strong Binder'],
            yticklabels=['Weak Binder', 'Strong Binder'])
ax4.set_xlabel('Predicted Label', fontweight='bold')
ax4.set_ylabel('True Label', fontweight='bold')
ax4.set_title(f'Confusion Matrix - {best_model_name}', fontweight='bold')

plt.tight_layout()
plt.savefig('ml_model_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Model performance visualization saved to: ml_model_performance.png")

# ====================================================================
# STEP 10: CROSS-VALIDATION VISUALIZATION
# ====================================================================

print("\n" + "=" * 70)
print("STEP 10: CROSS-VALIDATION ANALYSIS VISUALIZATION")
print("=" * 70)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cross-Validation Results', fontsize=14, fontweight='bold')

# Box plot of CV scores
ax1 = axes[0]
cv_scores_list = [cv_results[model] for model in cv_results.keys()]
bp = ax1.boxplot(cv_scores_list, labels=list(cv_results.keys()), patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax1.set_ylabel('ROC-AUC Score', fontweight='bold')
ax1.set_title('Cross-Validation Score Distribution', fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(alpha=0.3, axis='y')

# Mean CV scores with error bars
ax2 = axes[1]
model_names = list(cv_results.keys())
mean_scores = [cv_results[m].mean() for m in model_names]
std_scores = [cv_results[m].std() for m in model_names]

ax2.bar(range(len(model_names)), mean_scores, yerr=std_scores, capsize=10,
        color=colors, alpha=0.7, edgecolor='black')
ax2.set_xticks(range(len(model_names)))
ax2.set_xticklabels(model_names, rotation=45, ha='right')
ax2.set_ylabel('Mean ROC-AUC Score', fontweight='bold')
ax2.set_title('Cross-Validation Mean Scores (±std)', fontweight='bold')
ax2.set_ylim([0, 1.05])
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ml_cross_validation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Cross-validation visualization saved to: ml_cross_validation.png")

# ====================================================================
# STEP 11: PREDICTIONS ON NEW/UNSEEN EPITOPES
# ====================================================================

print("\n" + "=" * 70)
print("STEP 11: MAKING PREDICTIONS ON EPITOPES")
print("=" * 70)

# Make predictions on test set with best model
best_model = models[best_model_name]
best_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]

# Create prediction dataframe
prediction_results = pd.DataFrame({
    'Peptide': features_df.iloc[X_test.index]['peptide'].values,
    'IEDB_Score': features_df.iloc[X_test.index]['iedb_score'].values,
    'True_Label': y_test.values,
    'Predicted_Probability': best_pred_proba,
    'Predicted_Label': best_model.predict(X_test_scaled)
})

prediction_results['Prediction_Confidence'] = prediction_results['Predicted_Probability'].apply(
    lambda x: f"{max(x, 1-x)*100:.1f}%"
)

# Sort by prediction probability
prediction_results = prediction_results.sort_values('Predicted_Probability', ascending=False)

print(f"\n🎯 Top 20 Predicted Strong Binders (Test Set):\n")
print(f"{'#':<3} {'Peptide':<12} {'IEDB Score':<15} {'Confidence':<15} {'True Label':<12}")
print("-" * 60)

for idx, (i, row) in enumerate(prediction_results.head(20).iterrows(), 1):
    true_label = "Strong" if row['True_Label'] == 1 else "Weak"
    print(f"{idx:<3} {row['Peptide']:<12} {row['IEDB_Score']:<15.6f} {row['Prediction_Confidence']:<15} {true_label:<12}")

# Save predictions
prediction_results.to_csv("ml_predictions_test_set.csv", index=False)
print(f"\n✅ Predictions saved to: ml_predictions_test_set.csv")

# ====================================================================
# STEP 12: MODEL INTERPRETATION & SUMMARY
# ====================================================================

print("\n" + "=" * 70)
print("STEP 12: MODEL INTERPRETATION & SUMMARY")
print("=" * 70)

print(f"\n🏆 BEST MODEL ANALYSIS:")
print(f"   • Model: {best_model_name}")
print(f"   • Training Accuracy: {best_model.score(X_train_scaled, y_train):.4f}")
print(f"   • Testing Accuracy: {best_model.score(X_test_scaled, y_test):.4f}")
print(f"   • ROC-AUC Score: {results_df[results_df['Model'] == best_model_name]['ROC-AUC'].values[0]:.4f}")

print(f"\n📊 MODEL GENERALIZATION:")
best_cv_scores = cv_results[best_model_name]
print(f"   • Cross-validation ROC-AUC: {best_cv_scores.mean():.4f} (±{best_cv_scores.std():.4f})")
print(f"   • Min CV score: {best_cv_scores.min():.4f}")
print(f"   • Max CV score: {best_cv_scores.max():.4f}")

print(f"\n🔑 TOP 5 MOST IMPORTANT FEATURES:")
for idx, (i, row) in enumerate(importance_df.head(5).iterrows(), 1):
    print(f"   {idx}. {row['Feature']}: {row['Avg_Importance']:.4f}")

print(f"\n📈 PREDICTIVE INSIGHTS:")
print(f"   • Strong binders in test set: {sum(y_test)}/{len(y_test)}")
print(f"   • Correctly identified strong binders: {sum((rf_pred == 1) & (y_test == 1))}/{sum(y_test)}")
print(f"   • Recall (Sensitivity): {recall_score(y_test, rf_pred):.4f}")
print(f"   • Precision: {precision_score(y_test, rf_pred, zero_division=0):.4f}")

print(f"\n✅ OUTPUT FILES GENERATED:")
print(f"   1. ml_model_performance.csv - Model metrics comparison")
print(f"   2. feature_importance.csv - Feature importance rankings")
print(f"   3. ml_predictions_test_set.csv - Predictions for test peptides")
print(f"   4. ml_model_performance.png - Performance visualization")
print(f"   5. ml_cross_validation.png - Cross-validation results")

print("\n" + "=" * 70)
print("MACHINE LEARNING ANALYSIS COMPLETE! 🎉")
print("=" * 70)

print("\nNext Steps:")
print("  1. ✅ Review model performance and feature importance")
print("  2. ✅ Compare predictions with experimental validation (if available)")
print("  3. ➡️  Write Notebook 5: Summary & Publication Preparation")
print("  4. ➡️  Prepare manuscript for submission to journal")
print("\nModel Ready for:")
print("  • Predicting epitopes for new protein sequences")
print("  • Vaccine design optimization")
print("  • Immunoinformatics research")
print("=" * 70)

MACHINE LEARNING MODEL FOR EPITOPE PREDICTION - NOTEBOOK 4

STEP 1: LOADING AND PREPARING DATA
✅ Loaded epitope data from GitHub

📊 Dataset Overview:
   • Total epitopes: 50
   • Columns: ['peptide', 'allele', 'score', 'percentile_rank']
⚠️ Docking results not found, using IEDB predictions only

STEP 2: FEATURE ENGINEERING - CALCULATING PHYSICOCHEMICAL PROPERTIES

🧬 Calculating physicochemical features for 50 peptides...
✅ Features calculated for all peptides

📋 Feature Summary:
   • Total features: 18
   • Feature list: ['length', 'hydrophobic_aa', 'aromatic_aa', 'polar_aa', 'charged_aa', 'positive_charged', 'negative_charged', 'avg_hydrophobicity', 'max_hydrophobicity', 'min_hydrophobicity', 'net_charge', 'avg_charge', 'total_mass', 'avg_mass', 'aromaticity_ratio', 'hydrophobicity_ratio', 'mhc_p2_anchor', 'mhc_p9_anchor']

⚠️ Docking data not available, using IEDB score for binding prediction

STEP 3: CREATING TARGET VARIABLE - BINDER CLASSIFICATION

📊 Target Variable Distribution:
 

KeyError: 0